# BankToBook — Phased Regression Harness

**Issue #40 — P0 Baseline (B1–B5)**

Baseline using existing `llcBankView` / `llcExpRev` — no agents.  
Invariant: trial balance total = 0 at every phase.

| Cell | Purpose |
|------|------|
| B1 | Setup + config |
| B2 | Parse 2025 WF CSVs → bank DataFrame |
| B3 | Load llcExpRev (53 already-booked records) → ExpRev DataFrame |
| B4 | Double-entry GL expansion → GL DataFrame |
| B5 | Trial balance (Debit total − Credit total must = 0) |

P1+ cells will be added below as IngestAgent and BankAgent phases are built.

In [ ]:
# B1 — Setup & Config
import sys
import json
import datetime
import pandas as pd
import numpy as np
from pathlib import Path
from IPython.display import display

REPO = Path.cwd().parent          # llcRentalTracker/
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

from ledger import setup_paths
setup_paths.load_config('WBGroupLLC', 2025)

from ledger.LLC import LLC
llc = LLC('WBGroupLLC')

print("BUS root    :", setup_paths.TOP)
print("BankStmts   :", setup_paths.BANK_STMTS)
print("Accts dir   :", setup_paths.ACCTS_DIR)
print("Year        :", setup_paths.YEAR)

In [ ]:
# B2 — Parse 2025 WF CSVs → bank DataFrame
from ui.llcBankView import _parse_wf_csv

bank_dir = setup_paths.BANK_STMTS
csv_files = sorted([f for f in bank_dir.iterdir() if f.suffix.lower() == '.csv'])
print(f"CSV files found: {len(csv_files)}")
for f in csv_files:
    print(f"  {f.name}")

all_bank_rows = []
for csv_file in csv_files:
    with open(csv_file, 'r', encoding='utf-8', errors='replace') as fh:
        rows = _parse_wf_csv(fh.read())
    for row in rows:
        row['_source_csv'] = csv_file.name
    all_bank_rows.extend(rows)
    print(f"  {csv_file.name}: {len(rows)} rows parsed")

bank_df = pd.DataFrame(all_bank_rows).drop_duplicates(subset=['dt', 'amt', 'aType', 'desc'])
print(f"\nTotal bank rows (after dedup): {len(bank_df)}")
display(bank_df.head(5))

In [ ]:
# B3 — Load llcExpRev (already-booked 2025 records) → ExpRev DataFrame
from ledger.llcExpRev import llcExpRev

er_obj = llcExpRev(llc)
er_records = er_obj.load()          # always returns list (unwraps new dict format)
er_log     = er_obj.log_history()   # LogHistory for audit trail

print(f"ExpRev records : {len(er_records)}")
print(f"LogHistory entries : {len(er_log)}")

er_df = pd.DataFrame(er_records)
print(f"\nAccounts (acct):\n  {sorted(er_df['acct'].unique())}")
print(f"\nrefDB values: {sorted(er_df['refDB'].unique())}")
display(er_df[['dt', 'acct', 'Ledger', 'aType', 'amt', 'desc', 'propNm', 'refDB']].head(5))

In [ ]:
# B4 — Double-entry GL expansion
#
# Each llcExpRev record has two account legs:
#   acct   — primary account (e.g. Acct.Cash.Bank)
#   Ledger — contra account (e.g. Acct.Equity.Owner.Capital.Funds)
#
# Expansion: 2 GL rows per source record.
#   Side A: acct=acct,   aType=original aType
#   Side B: acct=Ledger, aType=flipped

def to_double_entry(records):
    df = pd.DataFrame(records) if not isinstance(records, pd.DataFrame) else records.copy()
    # Drop rows with missing Ledger (not a full dual-entry record)
    df = df[df['Ledger'].notna() & ~df['Ledger'].isin(['', 'nan'])].copy()

    side_a = df.copy()
    side_a['_side'] = 'A'

    side_b = df.copy()
    side_b['acct'] = side_b['Ledger']
    side_b['aType'] = side_b['aType'].apply(lambda v: 'Credit' if str(v).strip().lower() in ('debit', 'dr') else 'Debit')
    side_b['_side'] = 'B'

    gl = pd.concat([side_a, side_b], ignore_index=True)
    gl = gl.drop(columns=['Ledger', '_side'], errors='ignore')
    gl['signed_amt'] = gl.apply(
        lambda r: float(r['amt']) if str(r['aType']).strip().lower() in ('debit', 'dr') else -float(r['amt']),
        axis=1
    )
    return gl.sort_values('dt').reset_index(drop=True)


er_gl_df = to_double_entry(er_records)
print(f"Source records : {len(er_records)}")
print(f"GL rows (×2)   : {len(er_gl_df)}")
display(er_gl_df[['dt', 'acct', 'aType', 'amt', 'signed_amt', 'desc']].head(8))

In [ ]:
# B5 — Trial Balance  (INVARIANT: net signed amount = 0)
#
# Debit  = positive (increases asset / expense accounts)
# Credit = negative (increases liability / equity / income accounts)
# Net must be 0 — any non-zero value is a double-entry bug.

debit_total  = er_gl_df.loc[er_gl_df['aType'].str.lower() == 'debit',  'amt'].sum()
credit_total = er_gl_df.loc[er_gl_df['aType'].str.lower() == 'credit', 'amt'].sum()
net          = round(debit_total - credit_total, 2)

print(f"Total Debits  : ${debit_total:>12,.2f}")
print(f"Total Credits : ${credit_total:>12,.2f}")
print(f"Net (D − C)   : ${net:>12,.2f}")

if abs(net) < 0.01:
    print("\n✓ TRIAL BALANCE = 0  —  books are balanced (P0 baseline PASS)")
else:
    print(f"\n✗ TRIAL BALANCE ERROR: ${net:,.2f}  —  double-entry bug, fix before P1")

# Summary by account type
try:
    from ledger.llcCOA import ChartOfAccounts
    coa = ChartOfAccounts(llc)
    er_gl_df['acctType'] = er_gl_df['acct'].apply(lambda a: coa._Type(a) if a else '')
except Exception as e:
    print(f"(COA lookup skipped: {e})")
    er_gl_df['acctType'] = er_gl_df['acct'].apply(
        lambda a: 'Expense' if 'Exp' in str(a)
        else ('Income' if 'Rev' in str(a)
        else ('Asset' if ('Cash' in str(a) or 'Fixed' in str(a))
        else ('Equity' if 'Equity' in str(a)
        else 'Other')))
    )

tb = (
    er_gl_df.groupby(['acctType', 'aType'])['amt']
    .sum()
    .unstack(fill_value=0)
)
for col in ['Debit', 'Credit']:
    if col not in tb:
        tb[col] = 0
tb['Balance'] = tb['Debit'] - tb['Credit']
tb.loc['TOTAL'] = tb.sum()

print("\nTrial Balance by Account Type:")
display(tb.style.format('${:,.2f}'))

---
## P1 cells — IngestAgent compat check
_To be added after IngestAgent (BkVendorKB + BkTxnTypeDetector) is implemented._

- **P1a**: IngestAgent compat check — 2025 classify diff vs B3 baseline (expected: zero diff for Tier 1 rules)
- **P1b**: IngestAgent 2026 classify — new transactions

## P2 cells — BankAgent compat check
_To be added after BankAgent (BankCSVParser + BkDuplicateDetector + BkCIPGuard) is implemented._

- **P2a**: BankAgent 2025 compat — trial-balance delta vs B5 must be 0 for non-CIP rows
- **P2b**: BankAgent 2026 preview
- **P3a**: 2026 GL + trial balance
- **P3b**: optional commit cell (commented out by default)